In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, date
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import VECM
from scipy.linalg import eig
print(f"modules have been imported.")

In [ ]:
# Step 1: to simulate dummy macroeconomic data for 4 economies (T=100 periods)

np.random.seed(42)

num_periods = 100
economies = ['SG','US','CN','EU']
variables = ['gdp', 'inf', 'int']

columns = [f"{eco}_{var}" for eco in economies for var in variables]
data = pd.DataFrame(np.random.randn(num_periods, len(columns)), columns = columns)
display(data)

# 2. define fixed trade weights matrix (rows sum to 1, diags to 0)
# Rows: SG, US, CN, EU | Cols: SG, US, CN, EU

W = np.array([
    [0.0, 0.3, 0.4, 0.3], # SG trade partners
    [0.1, 0.0, 0.5, 0.4], # US trade partners
    [0.2, 0.4, 0.0, 0.4], # CN trade partners
    [0.2, 0.4, 0.4, 0.0]  # EU trade partners
])

W

In [ ]:
# 3. Construct Foreign Variables (X*) for Singapore (SG) as an example
sg_partners_data = data[[col for col in data.columns if not col.startswith('SG')]]
sg_partners_data

# reshape partners data to compute weighted average easily.
sg_w = W[0,1:]  # weights for US, CN, EU
sg_w

gdp_star = data['US_gdp'] * sg_w[0] + data['CN_gdp'] * sg_w[1] + data['EU_gdp'] * sg_w[2]
inf_star = data['US_inf'] * sg_w[0] + data['CN_inf'] * sg_w[1] + data['EU_inf'] * sg_w[2]
int_star = data['US_int'] * sg_w[0] + data['CN_int'] * sg_w[1] + data['EU_int'] * sg_w[2]

X_sg_domestic = data[['SG_gdp', 'SG_inf', 'SG_int']]
X_sg_foreign = pd.DataFrame({'gdp_star' : gdp_star,
                            'inf_star' : inf_star,
                            'int_star': int_star})

print(f"X_sg_domestic is: {display(X_sg_domestic)}")
print(f"X_sg_foreign is: {display(X_sg_foreign)}")


In [ ]:
# 4. Fit VARX(1) for Singapore
# Note: Treat contemporaneous foreign variables and their lags as exog
exog_data = pd.concat([X_sg_foreign, X_sg_foreign.shift(1)], axis = 1).dropna()
exog_data.columns = ['gdp_star', 'inf_star', 'int_star', 'gdp_star_lag1', 'inf_star_lag1', 'int_star_lag1']
endog_data = X_sg_domestic.iloc[1:] # Align time index due to lag shift.

print(f"X_sg_foreign is: {display(X_sg_foreign)}")
print(f"X_sg_foreign.shift(1) is: {display(X_sg_foreign.shift(1).dropna())}")
print(f"exog_data is: {display(exog_data)}")

varx_model = VAR(endog_data, exog=exog_data)
varx_results = varx_model.fit(maxlags = 1)
print(varx_results.summary())

In [ ]:
# Create Johansens test

# 1. Simulate a system where 2 variables are explicitly cointegrated (share a trend)
np.random.seed(42)
T = 200

# Common non-stationary stochastic trend (Random Walk)
common_trend = np.cumsum(np.random.normal(0,1,T))
common_trend

# Generate 3 long-term variables (z_t)
z1 = common_trend + np.random.normal(0,0.5, T) # Driven by trend
z2 = 0.8 * common_trend + np.random.normal(0, 0.5, T) # Cointegrated with z1
z3 = np.cumsum(np.random.normal(0, 1, T)) # Independent random walk

Z = np.vstack([z1, z2, z3]).T # shape: (T, 3)
print(f"Z matrix is: {Z}")
print(f"Z shape is: {Z.shape}")

# 2. prepare the VECM Data Arrays
# Z_t (current levels), Z_lag (lagged levels), dZ (differences)
dZ = np.diff(Z, axis = 0)  # \Delta z_t
Z_lag = Z[:-1, :]  # z_{t-1}
print(f"dZ matrix is: {dZ}")
print(f"dZ shape is: {dZ.shape}")
print(f"Z_lag matrix is: {Z_lag}")
print(f"Z_lag shape is: {Z_lag.shape}")

T_effective = dZ.shape[0] # no of rows (199)
k = Z.shape[1] # no. of variables (3).

In [ ]:
# 3. Compute Moment Matrices (Cross-products)
# M_00 = Variance of short-run changes
# M_11 = Variance of long-run lagged levels
# M_01 / M_10: Covariance between short-run changes and long run levels.
M_00 = (dZ.T @ dZ) / T_effective
M_11 = (Z_lag.T @ Z_lag) / T_effective
M_01 = (dZ.T @ Z_lag) / T_effective
M_10 = M_01.T

print(f"M_00, M_11, M_01 and M_10 are as follows:")
print(M_00, M_11, M_01, M_10)
print(M_00.shape)

In [ ]:
# 4. Solve Johansen's Generalized Eigenvalue Problem
# We solve: | \lambda * M_11 - M_10 * (M_00^-1) * M_01 | = 0
M_00_inv = np.linalg.inv(M_00)
A = M_10 @ M_00_inv @ M_01
B = M_11
print(f"M_00_inv is: {M_00_inv}")
print(f"A is: {A}")
print(f"B is: {B}")

# Compute generalized eigenvalues and eigenvectors 
eigenvalues, eigenvectors = eig(A, B)
print(f"eigenvalues, eigenvectors are:\n {eigenvalues},\n {eigenvectors}")

# Real components only, then sort descending by eigenvalue strength
idx = np.argsort(eigenvalues.real)[::-1]
eigenvalues = eigenvalues.real[idx]
eigenvectors = eigenvectors.real[:,idx] # to rearrange the eigenvectors based on the sorting of the eigenvalues computed earlier.
print(f"idx is: {idx}")
print(f"eigenvalues is:\n {eigenvalues}")
print(f"eigenvectors is:\n {eigenvectors}")

In [ ]:
print(f"--- Step 1: Johansen Sorted Eigenvalues ---")
for i, lam in enumerate(eigenvalues):
    print(f"Eigenvalue \\lambda_{i+1}: {lam:.4f}")

In [ ]:
# 5. Extract the Cointegrating Vector (Beta)
# the first eigenvector corresponding to the largest eigenvalue forms our main equilibrium vector
beta_1 = eigenvectors[:,0]
print(beta_1)

# Normalize the vector relative to the first variable for easy reading.
beta_1_normalized = beta_1 / beta_1[0]
print(beta_1_normalized)

print(f"\n--- Step 2: Cointegrating Vector (\\u03b2_1)---")
print(f"Normalized \\u03b2 vector: {beta_1_normalized}")
print(f"Long-run stationary relation: ({beta_1_normalized[0]:.2f} * z1) + ({beta_1_normalized[1]:.2f} * z2) + ({beta_1_normalized[2]:.2f} * z3) = Stationary Equilibrium")

# 6. Calculate Speed of adjustment (\u03b1)
# \u03b1 = M_01 * \u03b2 * (\u03b2' * M_11 * \u03b2)^-1
alpha_1 = M_01 @ beta_1 / (beta_1.T @ M_11 @ beta_1)
print("\n-- Step 3: Speed of Adjustment (\\uo3b1) ---")
for i, alp in enumerate(alpha_1):
    print(f"Economy variable z{i+1} correction speed: {alp:.4f}")